# 11 — Tool Calling and Tool Interface Design

## Scenario
Northstar can read order status or draft a refund request, but it cannot execute a refund directly. 
We need to give the model the ability to trigger a function in our application code to look up an order.

**The Concept:** Tool Calling (or Function Calling) is how LLMs interact with the outside world. The recorded response doesn't run the code; it outputs a structured JSON request (a `function_call`) asking *your application* to run the code.

## Step 1: Defining the Tool Interface

We define a standard Python function. The `google-genai` SDK automatically inspects the type hints and the docstring to generate the underlying JSON schema that the model understands.

## Step 2: Triggering the Tool Call

We pass the tool to the model and ask a question. Notice how we do **not** enable automatic execution. We want to inspect the manual loop to understand the security boundary.

## Step 3: Executing and Returning the Result

The application (us) now executes the actual Python code and sends the result back to the model so it can answer the user.

## Conclusion

By controlling the manual execution loop, the application maintains ultimate authority over security and authorization. If the tool was `execute_refund()`, the application could pause here, ask a human for approval, and only then return the `function_response` to the model.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab11 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: Defining the Tool Interface

In [ ]:
print(GET_ORDER_STATUS.model_dump_json(indent=2))
assert GET_ORDER_STATUS.name == "get_order_status"
assert OrderStatusArgs.model_validate({"order_id": "ORD-999"})

## Step 2: Triggering the Tool Call

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i11/call/ord-999")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED TOOL CALL:", response.tool_calls)
assert response.tool_calls[0].name == "get_order_status"

## Step 3: Executing and Returning the Result

In [ ]:
counter = [0]
principal = Principal(user_id="USER-0001", tenant="tenant-synthetic-a", roles={"support_agent"})
trace = run_tool_flow(client, "ord-999", principal, execution_counter=counter)
print("TOOL RESULT:", trace["tool_result"])
print("FINAL RESPONSE:", trace["final"])
assert trace["tool_result"]["result"] == "Processing — synthetic status"
assert counter == [1]
denied = run_tool_flow(client, "ord-777", principal, execution_counter=counter)
assert denied["tool_result"]["error"] == "not_authorized"

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.